# Data Cleaning and Integration Pipeline
This notebook handles the cleaning, standardizing, and merging of scraped automobile datasets, consolidating split source tracking files into unified CSV outputs.

In [2]:
# Import libraries for data manipulation, regular expressions, and file system tasks
import pandas as pd
import re
from pprint import pprint
import json
from time import sleep
import numpy as np
import jmespath
from functools import reduce
import os

## Load Datasets
Load initial scraped CSV files tracking vehicle manufacturer brands and basic attributes.

In [3]:
# Load brands metadata and structural basic details for crawled vehicles
brands = pd.read_csv("brands.csv")
cars = pd.read_csv("cars_basic_info.csv")

## Inspect Data Frames
Review data columns, structural alignment, and count configurations across missing entries.

In [4]:
# Check information layout and missing types for columns in the brands dataset
brands.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Brand ID    244 non-null    int64  
 1   Brands      244 non-null    object 
 2   logo        116 non-null    object 
 3   foundation  118 non-null    float64
 4   country     118 non-null    object 
dtypes: float64(1), int64(1), object(3)
memory usage: 9.7+ KB


In [5]:
# Check schema details and observation density across basic car entities
cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1196 entries, 0 to 1195
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Car ID      1196 non-null   int64 
 1   Name        1196 non-null   object
 2   Production  1196 non-null   object
 3   New Price   1196 non-null   object
 4   Used Price  1196 non-null   object
 5   Image URL   1196 non-null   object
dtypes: int64(1), object(5)
memory usage: 56.2+ KB


## Standardizing Foreign Keys
Construct and apply a text formatting standardizer to align vehicle models with parent manufacturing identifiers seamlessly.

In [6]:
def brand_standardizer(brand):
    """
    Standardizes vehicle names and strings into consistent lookup keys
    by targeting specific multi-word variants or extracting root tokens.
    """
    if "AC Schnitzer" in brand:
        return "AC2" 
    elif brand[:2] == "AC":
        return "AC1" 
    elif "De Lorean" in brand:
        return "De1" 
    elif "De Tomaso" in brand:
        return "De2" 
    else: 
        return brand.split(" ")[0]

In [7]:
# Generate cross-reference join codes via elements application across frames
brands['brand_code'] = brands['Brands'].apply(brand_standardizer)
cars['brand_code'] = cars['Name'].apply(brand_standardizer)

## Verify Structural Mapping Codes
Print sample extractions to guarantee customized edge cases are parsing uniformly across inputs.

In [8]:
# Query assigned brand alignments across complex identifiers
print(brands[["Brands", "brand_code"]][brands['brand_code'] == "AC1"])
print("-"*50)
print(cars[["Name", "brand_code"]][cars['brand_code'] == "AC1"])
print("+"*50)
print("+"*50)

print(brands[["Brands", "brand_code"]][brands['brand_code'] == "AC2"])
print("-"*50)
print(cars[["Name", "brand_code"]][cars['brand_code'] == "AC2"])
print("+"*50)
print("+"*50)

print(brands[["Brands", "brand_code"]][brands['brand_code'] == "De1"])
print("-"*50)
print(cars[["Name", "brand_code"]][cars['brand_code'] == "De1"])
print("+"*50)
print("+"*50)

print(brands[["Brands", "brand_code"]][brands['brand_code'] == "De2"])
print("-"*50)
print(cars[["Name", "brand_code"]][cars['brand_code'] == "De2"])

## Preview Data Classes
Validate first observations before unifying the brand profiles and vehicle logs.

In [9]:
# View top entries inside updated cars dataframe
cars.head()

,Car ID,Name,Production,New Price,Used Price,Image URL,brand_code
0,303,9FF 9F-V4002004 - 2007,2004 - 2007,£n/a,£100000 - £100000,https://www.supercarworld.com/cgi-bin/showcarp...,9FF
1,595,9FF GT9 Vmax2012 - date,2012 - date,£410000,£n/a - £n/a,https://www.supercarworld.com/cgi-bin/showcarp...,9FF
2,441,9FF GT9-R2008,2008,£n/a,£650000 - £650000,https://www.supercarworld.com/cgi-bin/showcarp...,9FF
3,594,9FF G-Tronic2012 - date,2012 - date,£213000,£n/a - £n/a,https://www.supercarworld.com/cgi-bin/showcarp...,9FF
4,723,ABT Audi R8 V102017 - date,2017 - date,£150000,£n/a - £n/a,https://www.supercarworld.com/cgi-bin/showcarp...,ABT


In [10]:
# View top entries inside updated brands dataframe 
brands.head()

,Brand ID,Brands,logo,foundation,country,brand_code
0,1,9FF,https://www.carlogos.org/logo/9ff-logo-2560x14...,2001.0,Germany,9FF
1,2,ABT,NaN,NaN,NaN,ABT
2,3,AC,https://www.carlogos.org/logo/AC-logo-1920x108...,1901.0,United Kingdom,AC1
3,4,AC Schnitzer,NaN,NaN,NaN,AC2
4,5,AIM,NaN,NaN,NaN,AIM


## Combine Vehicle Data Matrices
Join the two primary components via the common standardized code mapping context.

In [11]:
# brands_cars_merge = cars.join(brands,on="brand_code")
# brands_cars_merge.head()
# Left join datasets to capture metadata fields for every matching vehicle code
brands_cars_merge = pd.merge(cars, brands, on="brand_code", how="left")
brands_cars_merge.head()

,Car ID,Name,Production,New Price,Used Price,Image URL,brand_code,Brand ID,Brands,logo,foundation,country
0,303,9FF 9F-V4002004 - 2007,2004 - 2007,£n/a,£100000 - £100000,https://www.supercarworld.com/cgi-bin/showcarp...,9FF,1,9FF,https://www.carlogos.org/logo/9ff-logo-2560x14...,2001.0,Germany
1,595,9FF GT9 Vmax2012 - date,2012 - date,£410000,£n/a - £n/a,https://www.supercarworld.com/cgi-bin/showcarp...,9FF,1,9FF,https://www.carlogos.org/logo/9ff-logo-2560x14...,2001.0,Germany
2,441,9FF GT9-R2008,2008,£n/a,£650000 - £650000,https://www.supercarworld.com/cgi-bin/showcarp...,9FF,1,9FF,https://www.carlogos.org/logo/9ff-logo-2560x14...,2001.0,Germany
3,594,9FF G-Tronic2012 - date,2012 - date,£213000,£n/a - £n/a,https://www.supercarworld.com/cgi-bin/showcarp...,9FF,1,9FF,https://www.carlogos.org/logo/9ff-logo-2560x14...,2001.0,Germany
4,723,ABT Audi R8 V102017 - date,2017 - date,£150000,£n/a - £n/a,https://www.supercarworld.com/cgi-bin/showcarp...,ABT,2,ABT,NaN,NaN,NaN


In [12]:
# Confirm tail elements of consolidated output structural state
brands_cars_merge.tail()

,Car ID,Name,Production,New Price,Used Price,Image URL,brand_code,Brand ID,Brands,logo,foundation,country
1191,698,Zenos E10R2016 - date,2016 - date,£40000,£25000 - £30000,https://www.supercarworld.com/cgi-bin/showcarp...,Zenos,243,Zenos,https://www.carlogos.org/logo/Zenos-Cars-logo-...,2012.0,United Kingdom
1192,1149,Zenvo Aurora2025 - date,2025 - date,£2000000,£n/a - £n/a,https://www.supercarworld.com/cgi-bin/showcarp...,Zenvo,244,Zenvo,https://www.carlogos.org/logo/Zenvo-logo-2009-...,2009.0,Denmark
1193,497,Zenvo ST12009 - 2016,2009 - 2016,£660000,£n/a - £n/a,https://www.supercarworld.com/cgi-bin/showcarp...,Zenvo,244,Zenvo,https://www.carlogos.org/logo/Zenvo-logo-2009-...,2009.0,Denmark
1194,732,Zenvo TS1 GT2017 - date,2017 - date,£n/a,£n/a - £n/a,https://www.supercarworld.com/cgi-bin/showcarp...,Zenvo,244,Zenvo,https://www.carlogos.org/logo/Zenvo-logo-2009-...,2009.0,Denmark
1195,943,Zenvo TSR-S2018 - date,2018 - date,£2000000,£1100000 - £1300000,https://www.supercarworld.com/cgi-bin/showcarp...,Zenvo,244,Zenvo,https://www.carlogos.org/logo/Zenvo-logo-2009-...,2009.0,Denmark


## Drop Intermediate Columns and Fix Schema Fields
Prune lookup helpers and adjust duplicate system labels inside the master matrix.

In [13]:
# Remove temporary join token and fix naming syntax variations if any are created
brands_cars_merge.drop(columns=["brand_code"], inplace=True)
brands_cars_merge.rename(columns={"Brand ID_y": "Brand ID"}, inplace=True)

In [14]:
# Track properties and type structures across the comprehensive merge model
brands_cars_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1196 entries, 0 to 1195
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Car ID      1196 non-null   int64  
 1   Name        1196 non-null   object 
 2   Production  1196 non-null   object 
 3   New Price   1196 non-null   object 
 4   Used Price  1196 non-null   object 
 5   Image URL   1196 non-null   object 
 6   Brand ID    1196 non-null   int64  
 7   Brands      1196 non-null   object 
 8   logo        1026 non-null   object 
 9   foundation  1024 non-null   float64
 10  country     1024 non-null   object 
dtypes: float64(1), int64(2), object(8)
memory usage: 102.9+ KB


## Inspect Original Data Frame Schemas
Assess the current state of source frames to establish accurate alignment criteria.

In [15]:
# Recheck dimensions and indices inside primary cars variable matrix
cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1196 entries, 0 to 1195
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Car ID      1196 non-null   int64 
 1   Name        1196 non-null   object
 2   Production  1196 non-null   object
 3   New Price   1196 non-null   object
 4   Used Price  1196 non-null   object
 5   Image URL   1196 non-null   object
 6   brand_code  1196 non-null   object
dtypes: int64(1), object(6)
memory usage: 65.5+ KB


In [16]:
# Recheck configuration map status for parent brand records
brands.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Brand ID    244 non-null    int64  
 1   Brands      244 non-null    object 
 2   logo        116 non-null    object 
 3   foundation  118 non-null    float64
 4   country     118 non-null    object 
 5   brand_code  244 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 11.6+ KB


## Project Subsets and Deduplicate Records
Isolate distinct normalized datasets to prevent cross-join duplication anomalies.

In [17]:
# list(brands.columns), list(cars.columns)
# Define targets to extract isolated entities from merged components
brands_columns = ['Brand ID', 'Brands', 'logo', 'foundation', 'country']
cars_columns = ['Car ID','Name','Production','New Price','Used Price','Image URL','Brand ID']

In [18]:
# Slice structural manufacturer details out from consolidated data layer
brands_ = brands_cars_merge[brands_columns]
brands_.head()

,Brand ID,Brands,logo,foundation,country
0,1,9FF,https://www.carlogos.org/logo/9ff-logo-2560x14...,2001.0,Germany
1,1,9FF,https://www.carlogos.org/logo/9ff-logo-2560x14...,2001.0,Germany
2,1,9FF,https://www.carlogos.org/logo/9ff-logo-2560x14...,2001.0,Germany
3,1,9FF,https://www.carlogos.org/logo/9ff-logo-2560x14...,2001.0,Germany
4,2,ABT,NaN,NaN,NaN


In [19]:
# Extract foundational vehicle metrics containing relational key associations
cars_ = brands_cars_merge[cars_columns]
cars_.head()

,Car ID,Name,Production,New Price,Used Price,Image URL,Brand ID
0,303,9FF 9F-V4002004 - 2007,2004 - 2007,£n/a,£100000 - £100000,https://www.supercarworld.com/cgi-bin/showcarp...,1
1,595,9FF GT9 Vmax2012 - date,2012 - date,£410000,£n/a - £n/a,https://www.supercarworld.com/cgi-bin/showcarp...,1
2,441,9FF GT9-R2008,2008,£n/a,£650000 - £650000,https://www.supercarworld.com/cgi-bin/showcarp...,1
3,594,9FF G-Tronic2012 - date,2012 - date,£213000,£n/a - £n/a,https://www.supercarworld.com/cgi-bin/showcarp...,1
4,723,ABT Audi R8 V102017 - date,2017 - date,£150000,£n/a - £n/a,https://www.supercarworld.com/cgi-bin/showcarp...,2


In [20]:
# Compare record length to guarantee preservation of data entities
len(cars_), len(cars)

(1196, 1196)

In [21]:
# Observe inflated counts inside slicing before executing unique filtering steps
len(brands_), len(brands)

(1196, 244)

In [22]:
# Prune duplicate instances within slices to retain exclusive manufacturer records
brands_.drop_duplicates(subset="Brand ID", inplace= True)

C:\Users\Abdalrhman\AppData\Local\Temp\ipykernel_17760\2268836528.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  brands_.drop_duplicates(subset="Brand ID", inplace= True)


In [23]:
# Validate corrected record matching length metrics 
len(brands_), len(brands)

(244, 244)

## Relational ID Diagnostics
Verify primary data integrity configurations on parsed IDs across clean subsets.

In [24]:
# Print diagnostic references across target brands to cross-check relationship correctness
print(brands_[["Brands", "Brand ID"]][brands_["Brand ID"] == 3])
print("\n")
print(cars_[["Name", "Brand ID"]][cars_["Brand ID"] == 3])
print("+"*50)
print("+"*50)

print(brands_[["Brands", "Brand ID"]][brands_["Brand ID"] == 4])
print("\n")
print(cars_[["Name", "Brand ID"]][cars_["Brand ID"] == 4])
print("+"*50)
print("+"*50)

print(brands_[["Brands", "Brand ID"]][brands_["Brand ID"] == 62])
print("\n")
print(cars_[["Name", "Brand ID"]][cars_["Brand ID"] == 62])
print("+"*50)
print("+"*50)

print(brands_[["Brands", "Brand ID"]][brands_["Brand ID"] == 63])
print("\n")
print(cars_[["Name", "Brand ID"]][cars_["Brand ID"] == 63])

  Brands  Brand ID
5     AC         3


                          Name  Brand ID
5  AC 378 GT Zagato2012 - 2014         3
6      AC Cobra 4271965 - 1967         3
7    AC Superblower1997 - 2002         3
++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++
         Brands  Brand ID
8  AC Schnitzer         4


                                Name  Brand ID
8       AC Schnitzer ACS82017 - date         4
9  AC Schnitzer ACS8 5.0i2019 - date         4
++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++
        Brands  Brand ID
324  De Lorean        62


                            Name  Brand ID
324        De Lorean Alpha 52024        62
325  De Lorean DMC-121981 - 1986        62
++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++
        Brands  Brand ID
326  De Tomaso        63


                                 Name  Brand ID
326        De 

## Save Normalized Local Backups
Write out the cleaned relational base tables into CSVs.

In [25]:
# Update files dynamically without saving indexes
brands_.to_csv("brands.csv", index=False)
cars_.to_csv("cars_basic_info.csv", index=False)

## Comprehensive Feature Consolidation
Load all secondary scraped automotive specifications datasets to perform a master join across uniform entity IDs.

In [26]:
# Read secondary specification and performance sheets into target buffers
basic = pd.read_csv("cars_basic_info.csv")
dimensions = pd.read_csv("cars_dimensions.csv")
engines = pd.read_csv("cars_engine.csv")
general = pd.read_csv("cars_general.csv")
over_all = pd.read_csv("cars_overall_rating.csv")
performance = pd.read_csv("cars_performance.csv")
ratings = pd.read_csv("cars_rating.csv")

In [27]:
# Group all references into iterable list and merge via functional reduce operations
dfs = [basic, dimensions, engines, general, over_all, performance, ratings]

super_cars = reduce(lambda left, right: pd.merge(left, right, on="Car ID", how="inner"), dfs)

## Prune Master Feature Schema
Evaluate structural elements and wipe trailing or unformatted garbage tokens generated via raw extraction logs.

In [28]:
# Print diagnostic layout info regarding total columns across master structure
super_cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1196 entries, 0 to 1195
Data columns (total 63 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Car ID                              1196 non-null   int64  
 1   Name                                1196 non-null   object 
 2   Production                          1196 non-null   object 
 3   New Price                           1196 non-null   object 
 4   Used Price                          1196 non-null   object 
 5   Image URL                           1196 non-null   object 
 6   Brand ID                            1196 non-null   int64  
 7   Length                              818 non-null    object 
 8   Width                               791 non-null    object 
 9   Height                              778 non-null    object 
 10  Weight                              1012 non-null   object 
 11  Wheels (F/R)                        745 non

In [29]:
# Filter and inspect trailing unstructured attributes 
super_cars.columns.tolist()[-6:]

['Unnamed: 11',
 '33 Navajo (230 bhp)',
 'Boreas Hybrid (2017 - date)',
 'Wiesmann GT (2005 - 2014)',
 'Koenigsegg CCX (2005 - 2010)',
 'Pininfarina Battista (2022 - date)']

In [30]:
# Drop unformatted artifacts dynamically via positional list references
super_cars.drop(columns=super_cars.columns.tolist()[-6:], inplace=True)

In [31]:
# Verify target dimensionality attributes following cleanup metrics
super_cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1196 entries, 0 to 1195
Data columns (total 57 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Car ID             1196 non-null   int64  
 1   Name               1196 non-null   object 
 2   Production         1196 non-null   object 
 3   New Price          1196 non-null   object 
 4   Used Price         1196 non-null   object 
 5   Image URL          1196 non-null   object 
 6   Brand ID           1196 non-null   int64  
 7   Length             818 non-null    object 
 8   Width              791 non-null    object 
 9   Height             778 non-null    object 
 10  Weight             1012 non-null   object 
 11  Wheels (F/R)       745 non-null    object 
 12  Tyres (F/R)        614 non-null    object 
 13  Fuel Capacity      414 non-null    object 
 14  Type               1183 non-null   object 
 15  Details            1146 non-null   object 
 16  Capacity           1075 

## Final Export and Workspace Cleanup
Export consolidated output to the main master repository file, and programmatically purge obsolete segment matrices.

In [32]:
# Save aggregated feature dataset matrix
super_cars.to_csv("super_cars_data.csv", index=False)

In [33]:
# List split intermediate staging files
raw_cars_data = ["cars_basic_info.csv",
"cars_dimensions.csv",
"cars_engine.csv",
"cars_general.csv",
"cars_overall_rating.csv",
"cars_performance.csv",
"cars_rating.csv"]

# Purge redundant storage staging pieces from disk
for file in raw_cars_data:
    os.remove(file)